In [ ]:
# @title 1. 환경 설정 및 라이브러리 설치 (Cell 1)
"""
이 셀에서는 실습에 필요한 라이브러리들을 설치하고 임포트합니다.
과제의 NLP pipeline 예시를 이미지 데이터에 적용한 확장 실습입니다.
- transformers: Hugging Face 모델 및 파이프라인 사용을 위한 라이브러리
- datasets: Hugging Face Hub의 이미지 데이터셋 로드를 위한 라이브러리
- accelerate: GPU 환경에서 모델 로딩을 도와주는 라이브러리
- pillow: 이미지 데이터 처리
- torch: PyTorch 라이브러리 (기본 백엔드)
"""
!pip install -qU transformers datasets accelerate pillow matplotlib "pandas<3"

import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset, load_dataset
from transformers import pipeline
from IPython.display import display

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.unicode_minus'] = False

# GPU 사용 가능 여부 확인 및 설정 (Colab에서는 GPU 런타임 권장)
device = 0 if torch.cuda.is_available() else -1
print(f"사용 가능한 디바이스: {'GPU' if device == 0 else 'CPU'}")



각주 [1] 프리셋의 설치/임포트 셀을 유지하되, 사진 데이터 처리에 필요한 pillow, matplotlib, pandas만 추가했습니다.


In [ ]:
# @title 2. 데이터셋 로드 및 준비 (Cell 2)
"""
이 셀에서는 Hugging Face의 Flickr8k 이미지 데이터셋을 로드합니다.
전체 이미지 파일을 다운로드하지 않도록 streaming=True로 열고, 처음 10개만 Dataset으로 변환합니다.
'image' 컬럼이 분위기를 분석할 원본 사진입니다.
"""
# 실습/제출용으로 처음부터 작은 샘플만 로드
num_samples_to_use = 10
streamed_dataset = load_dataset('intro/flickr8k', split='test', streaming=True)
image_subset = Dataset.from_list(list(streamed_dataset.take(num_samples_to_use)))

print("로드된 데이터셋 정보:")
print(image_subset)
print(f"사용 샘플 수: {len(image_subset)}개")

print("\n첫 번째 데이터 예시 (image, caption 확인):")
display(image_subset[0]['image'])
print(image_subset[0]['caption_0'])

# 데이터셋 확인을 위해 Pandas DataFrame으로 변환 (이미지 컬럼은 제외)
df_check = pd.DataFrame(image_subset.remove_columns(['image']))
print("\n데이터셋 일부 미리보기 (DataFrame):")
display(df_check.head(3))



각주 [2] 프리셋의 데이터셋 로드 위치를 그대로 사용하고, KLUE 대신 Hugging Face 이미지 데이터셋 intro/flickr8k를 streaming 방식으로 10개만 가져오도록 바꿨습니다.


In [ ]:
# @title 3. 분위기 분석 모델 파이프라인 로드 (Cell 3)
"""
이 셀에서는 사진 분위기 분석을 위한 CLIP 기반 zero-shot image classification 파이프라인을 로드합니다.
모델: openai/clip-vit-base-patch32
파이프라인 타입: zero-shot-image-classification
"""
model_id = 'openai/clip-vit-base-patch32'

mood_label_map = {
    'bright and cheerful atmosphere': '밝고 경쾌한 분위기',
    'calm and peaceful atmosphere': '차분하고 평화로운 분위기',
    'warm and cozy atmosphere': '따뜻하고 아늑한 분위기',
    'dark and melancholic atmosphere': '어둡고 쓸쓸한 분위기',
    'lonely and quiet atmosphere': '고요하고 외로운 분위기',
    'romantic and dreamy atmosphere': '낭만적이고 몽환적인 분위기',
    'energetic and lively atmosphere': '활기차고 역동적인 분위기',
}

candidate_labels = list(mood_label_map.keys())

mood_classifier = pipeline(
    'zero-shot-image-classification',
    model=model_id,
    device=device,
)

print("분위기 분석 파이프라인 로드 완료.")
print("후보 라벨:")
for label in candidate_labels:
    print(f"- {mood_label_map[label]} ({label})")



각주 [3] 프리셋의 모델 파이프라인 로드 셀을 유지하되, 요약 모델 대신 CLIP zero-shot 이미지 분류 파이프라인을 로드했습니다.


In [ ]:
# @title 4. 사진 분위기 분석 및 데이터셋에 추가 (Cell 4)
"""
이 셀에서는 로드된 CLIP 파이프라인을 사용하여 각 이미지의 분위기를 분석합니다.
map 함수를 사용하여 데이터셋의 각 샘플에 분석 함수를 적용하고,
결과를 'mood_label_ko', 'mood_score' 등의 새로운 컬럼에 저장합니다.
"""
# 분위기 분석을 수행하는 함수 정의
def analyze_image_mood(example):
  """데이터셋의 'image'를 받아 분위기 분석 결과를 반환하는 함수"""
  predictions = mood_classifier(
      example['image'].convert('RGB'),
      candidate_labels=candidate_labels,
      hypothesis_template='a photo with a {}.',
  )
  top3 = predictions[:3]
  best = top3[0]

  example['mood_label'] = best['label']
  example['mood_label_ko'] = mood_label_map[best['label']]
  example['mood_score'] = round(float(best['score']), 4)
  example['top3_moods_ko'] = [mood_label_map[item['label']] for item in top3]
  example['top3_scores'] = [round(float(item['score']), 4) for item in top3]
  return example

print("사진 분위기 분석 작업을 시작합니다... (데이터 양에 따라 시간이 소요될 수 있습니다)")
mood_dataset = image_subset.map(analyze_image_mood, load_from_cache_file=False)
print("사진 분위기 분석 작업 완료.")

print("\n분위기 분석 결과가 추가된 데이터셋 정보:")
print(mood_dataset)

print("\n첫 번째 데이터의 캡션과 분위기 분석 결과:")
print("--- 캡션 ---")
print(mood_dataset[0]['caption_0'])
print("\n--- 분위기 ---")
print(mood_dataset[0]['mood_label_ko'], mood_dataset[0]['mood_score'])



각주 [4] 프리셋의 map() 적용 셀을 유지하고, summary 컬럼 대신 사진 분위기 분석 결과 컬럼을 추가하도록 함수만 바꿨습니다.


In [ ]:
# @title 5. 분위기 분석 결과 표 확인 (Cell 5)
"""
이 셀에서는 분위기 분석 결과를 표 형태로 확인합니다.
이미지 컬럼은 표에서 제외하고, 캡션과 예측 분위기 컬럼을 중심으로 확인합니다.
"""
result_df = mood_dataset.remove_columns(['image']).to_pandas()

result_columns = [
    'caption_0',
    'mood_label_ko',
    'mood_score',
    'top3_moods_ko',
    'top3_scores',
]

print("분위기 분석 결과 미리보기 (DataFrame):")
display(result_df[result_columns].head(10))



각주 [5] 프리셋의 중간 확인 셀 역할을 살려, 새로 추가된 분위기 컬럼을 DataFrame으로 확인합니다.


In [ ]:
# @title 6. 이미지와 분위기 결과 함께 확인 (Cell 6)
"""
이 셀에서는 이미지와 예측된 분위기 라벨을 함께 확인합니다.
분위기 분석은 주관성이 있으므로, 표뿐 아니라 실제 이미지를 같이 보는 과정이 필요합니다.
"""
def show_mood_examples(dataset, count=6):
  count = min(count, len(dataset))
  cols = 3
  rows = (count + cols - 1) // cols
  fig, axes = plt.subplots(rows, cols, figsize=(14, 4.5 * rows))
  axes = axes.flatten() if count > 1 else [axes]

  for ax, item in zip(axes, dataset.select(range(count))):
    ax.imshow(item['image'])
    ax.set_title(
        f"{item['mood_label_ko']}\nscore={item['mood_score']:.2f}",
        fontsize=11,
    )
    ax.axis('off')

  for ax in axes[count:]:
    ax.axis('off')

  plt.tight_layout()
  plt.show()

show_mood_examples(mood_dataset, count=6)



각주 [6] 이미지 과제라 표만으로는 검토가 부족해서, 실제 이미지와 분위기 라벨을 함께 확인하는 시각화만 추가했습니다.


In [ ]:
# @title 7. 분위기 라벨 분포 확인 (Cell 7)
"""
이 셀에서는 분석된 분위기 라벨이 샘플 안에서 어떻게 분포하는지 확인합니다.
"""
mood_counts = result_df['mood_label_ko'].value_counts().sort_values(ascending=True)

ax = mood_counts.plot(kind='barh', color='#4C78A8')
ax.set_title('사진 분위기 분석 결과 분포')
ax.set_xlabel('이미지 수')
ax.set_ylabel('분위기 라벨')
plt.tight_layout()
plt.show()



각주 [7] 라벨 분포를 확인해 모델이 특정 분위기로 치우치는지 간단히 점검합니다.


In [ ]:
# @title 8. 결과 CSV 저장 (Cell 8)
"""
이 셀에서는 최종 분석 결과를 CSV 파일로 저장합니다.
"""
output_csv = 'image_mood_analysis_results.csv'
result_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"결과를 CSV 파일로 저장했습니다: {output_csv}")



각주 [8] 결과 공유를 위해 CSV 저장 셀을 둔 것입니다. 모델 동작 자체에는 필수는 아닙니다.


In [ ]:
# @title 9. 결과 정리 및 마무리 (Cell 9)
"""
모든 단계를 거쳐 생성된 최종 데이터셋(mood_dataset)에는
원본 Flickr8k 데이터에 'mood_label', 'mood_label_ko', 'mood_score', 'top3_moods_ko', 'top3_scores' 컬럼이 추가되었습니다.
"""
print("모든 작업이 완료되었습니다.")
print("최종 데이터셋 컬럼:", mood_dataset.column_names)

# 최종 결과 확인 (첫 5개 샘플)
for i in range(min(5, len(mood_dataset))):
  print(f"\n--- 샘플 {i+1} ---")
  print(f"캡션: {mood_dataset[i]['caption_0']}")
  print(f"분위기 분석: {mood_dataset[i]['mood_label_ko']}")
  print(f"점수: {mood_dataset[i]['mood_score']}")
  print(f"상위 3개 후보: {mood_dataset[i]['top3_moods_ko']}")



각주 [9] 최종 Dataset에 추가된 컬럼을 확인하고, 과제 요구사항인 datasets, pipeline, map() 사용 결과를 정리합니다.
